[<img src="https://colab.research.google.com/assets/colab-badge.svg" align="left" alt="Open In Colab"/>](https://colab.research.google.com/github/luiscuellarh/apigee-unified-ai-gateway/blob/main/03_apigee_mcp_gateway.ipynb)

<br><br>


# Apigee AI Gateway — MCP Gateway Governance & Policies (AFT)

**Repository Location:** [`https://github.com/luiscuellarh/apigee-unified-ai-gateway/blob/main/03_apigee_mcp_gateway.ipynb`](https://github.com/luiscuellarh/apigee-unified-ai-gateway/blob/main/03_apigee_mcp_gateway.ipynb)  
**Execution Constraint:** This notebook is version-controlled in GitHub and designed to run cleanly when opened directly from that repository path in Google Colab (no local-only paths, no machine-specific assumptions, and no state carried between sessions).

**Framework & Toolkit:**
- **Toolkit:** **AFT (`aft` — Apigee Foundation Toolkit / Apigee Feature Templater)** (`https://github.com/apigee/apigee-templater`)
- **Repository:** [`https://github.com/luiscuellarh/apigee-unified-ai-gateway`](https://github.com/luiscuellarh/apigee-unified-ai-gateway)
- **Assumptions:** Apigee X is already provisioned with `APIGEE_ENV` attached to an environment group (`ENV_GROUP`), and target MCP backend services are reachable.

---

## Step 0 — Preflight (Single-Run Blocking Gate)
Run **Step 0** once to execute all four preflight stages (`0.1`–`0.4`) in a single execution:
- **0.1** Inputs & Global Configuration
- **0.2** Verify Apigee Topology, APIs & IAM
- **0.3** Verify Service Account Roles, Permissions & `<GoogleAccessToken>` Bindings + Live Gemini
- **0.4** Install AFT CLI & Preflight Summary Gate


In [ ]:
#@title 0.1–0.4 Step 0 Preflight (Inputs, Apigee Topology & IAM, Service Account & Live Gemini, AFT CLI & Summary Gate)
# Runs 0.1, 0.2, 0.3, and 0.4 sequentially in a single execution.
import json, os, pathlib, re, shutil, subprocess, time, urllib.error, urllib.request
import requests

# ============================================================================
# 0.1 Inputs & Global Configuration
# ============================================================================
print("▶ [0.1] Inputs & Global Configuration...")
try:
    from google.colab import auth
    auth.authenticate_user()
    print("✅ Authenticated successfully with Google Cloud in Colab.")
except ImportError:
    print("ℹ️ Running in local/standard Jupyter environment.")

#@markdown ### Project & Apigee
PROJECT_ID = "" #@param {type:"string"}
APIGEE_ORG = "" #@param {type:"string"}
APIGEE_ENV = "dev" #@param {type:"string"}
ENV_GROUP = "dev-group" #@param {type:"string"}
REGION = "europe-west1" #@param {type:"string"}
HOSTNAME = "" #@param {type:"string"}

#@markdown ### Service Account
SERVICE_ACCOUNT = "" #@param {type:"string"}

#@markdown ### Gemini
GEMINI_MODEL = "gemini-2.5-flash" #@param {type:"string"}

#@markdown ### Behaviour Flags
ENABLE_APIS = False #@param {type:"boolean"}

PROJECT_ID, APIGEE_ENV, ENV_GROUP, REGION, HOSTNAME, SERVICE_ACCOUNT = (
    v.strip() for v in (PROJECT_ID, APIGEE_ENV, ENV_GROUP, REGION, HOSTNAME, SERVICE_ACCOUNT)
)
APIGEE_ORG = APIGEE_ORG.strip() or PROJECT_ID
APIGEE_ENVIRONMENT = APIGEE_ENV
GEMINI_MODEL = GEMINI_MODEL.strip().removeprefix("google/")

missing = [k for k, v in {"PROJECT_ID": PROJECT_ID, "APIGEE_ENV": APIGEE_ENV, "REGION": REGION, "GEMINI_MODEL": GEMINI_MODEL}.items() if not v]
if missing:
    raise SystemExit(f"❌ HALT: Missing required parameter(s): {", ".join(missing)}.")

# If SERVICE_ACCOUNT is provided, do not ask for a name and do not create a new one
SA_PROVIDED = bool(SERVICE_ACCOUNT)
if SERVICE_ACCOUNT and "@" not in SERVICE_ACCOUNT:
    SERVICE_ACCOUNT = f"{SERVICE_ACCOUNT}@{PROJECT_ID}.iam.gserviceaccount.com"
if SERVICE_ACCOUNT and not re.fullmatch(r"[a-zA-Z0-9._-]+@[a-zA-Z0-9.-]+\.gserviceaccount\.com", SERVICE_ACCOUNT):
    raise SystemExit(f"❌ HALT: Invalid SERVICE_ACCOUNT '{SERVICE_ACCOUNT}' (expected *.gserviceaccount.com).")

os.environ.update({
    "GOOGLE_CLOUD_PROJECT": PROJECT_ID,
    "PROJECT_ID": PROJECT_ID,
    "APIGEE_ORG": APIGEE_ORG,
    "APIGEE_ENV": APIGEE_ENV,
    "APIGEE_ENVIRONMENT": APIGEE_ENVIRONMENT,
    "ENV_GROUP": ENV_GROUP,
    "REGION": REGION,
    "HOSTNAME": HOSTNAME,
    "SERVICE_ACCOUNT": SERVICE_ACCOUNT,
    "GEMINI_MODEL": GEMINI_MODEL,
})
subprocess.run(["gcloud", "config", "set", "project", PROJECT_ID, "--quiet"], check=True, capture_output=True)

PREFLIGHT = {"inputs": True, "apigee": False, "service_account": False, "gemini": False, "aft": False, "repo": False}
sa_mode = f"{SERVICE_ACCOUNT} (provided — verify only, no creation)" if SA_PROVIDED else f"apigee-ai-gateway-sa@{PROJECT_ID}.iam.gserviceaccount.com (auto-provision if missing)"
print(f"✅ [0.1] Inputs ready | project={PROJECT_ID} | org={APIGEE_ORG} | env={APIGEE_ENV} ({ENV_GROUP}) | region={REGION} | sa={sa_mode}")

# ============================================================================
# 0.2 Verify Apigee Topology, APIs & IAM
# ============================================================================
print("\n▶ [0.2] Verify Apigee Topology, APIs & IAM...")
def get_auth_token() -> str:
    return subprocess.check_output(["gcloud", "auth", "print-access-token"], text=True).strip()

def gcp_request(method: str, url: str, json_body=None, headers=None):
    req_headers = {"Authorization": f"Bearer {get_auth_token()}", "Content-Type": "application/json"}
    if headers:
        req_headers.update(headers)
    try:
        resp = requests.request(method, url, json=json_body, headers=req_headers, timeout=30)
        return resp.status_code, (resp.json() if resp.text else {})
    except ValueError:
        return resp.status_code, {"raw": resp.text}
    except Exception as exc:
        return 0, {"error": str(exc), "url": url}

def wait_for_proxy_ready(proxy_name: str, env_name: str = None, timeout_sec: int = 180) -> str:
    env_target = env_name or APIGEE_ENV
    list_url = f"{APIGEE_BASE}/environments/{env_target}/apis/{proxy_name}/deployments"
    deadline = time.time() + timeout_sec
    last_rev, last_state = None, "NOT_DEPLOYED"
    print(f"⏳ Waiting for proxy '{proxy_name}' to reach READY state in env '{env_target}'...")
    while time.time() < deadline:
        s_list, d_list = gcp_request("GET", list_url)
        deployments = d_list.get("deployments", []) if s_list == 200 else []
        if deployments:
            latest = sorted(deployments, key=lambda d: int(d.get("revision", 0)))[-1]
            last_rev = latest.get("revision")
            rev_url = f"{APIGEE_BASE}/environments/{env_target}/apis/{proxy_name}/revisions/{last_rev}/deployments"
            s_rev, d_rev = gcp_request("GET", rev_url)
            last_state = d_rev.get("state", latest.get("state", "UNKNOWN")) if s_rev == 200 else "UNKNOWN"
            if last_state == "READY":
                print(f"✅ Proxy '{proxy_name}' (revision {last_rev}) is READY in env '{env_target}'.")
                return str(last_rev)
            print(f"   • proxy={proxy_name} | rev={last_rev} | state={last_state} — waiting 5s...")
        else:
            print(f"   • proxy={proxy_name} | state=WAITING_FOR_DEPLOYMENT — waiting 5s...")
        time.sleep(5)
    raise SystemExit(
        f"❌ HALT: Timed out waiting for proxy '{proxy_name}' to become READY in env '{env_target}' "
        f"(last revision={last_rev}, state={last_state})."
    )

ACTIVE_IDENTITY = subprocess.check_output(["gcloud", "config", "get-value", "account"], text=True).strip()
if not ACTIVE_IDENTITY:
    _, t_info = gcp_request("GET", f"https://oauth2.googleapis.com/tokeninfo?access_token={get_auth_token()}")
    ACTIVE_IDENTITY = t_info.get("email", "")
if not ACTIVE_IDENTITY:
    raise SystemExit("❌ HALT: Could not resolve active caller identity.")
caller_member_prefix = "serviceAccount" if ACTIVE_IDENTITY.endswith(".gserviceaccount.com") else "user"
print(f"✅ Active identity: {ACTIVE_IDENTITY}")

# 1. Verify required GCP APIs
REQUIRED_SERVICES = ["apigee.googleapis.com", "cloudresourcemanager.googleapis.com", "iamcredentials.googleapis.com", "aiplatform.googleapis.com"]
disabled_svcs = [
    svc for svc in REQUIRED_SERVICES
    if gcp_request("GET", f"https://serviceusage.googleapis.com/v1/projects/{PROJECT_ID}/services/{svc}")[1].get("state") != "ENABLED"
]
if disabled_svcs:
    if ENABLE_APIS or disabled_svcs == ["iamcredentials.googleapis.com"]:
        gcp_request("POST", f"https://serviceusage.googleapis.com/v1/projects/{PROJECT_ID}/services:batchEnable", {"serviceIds": disabled_svcs})
        time.sleep(3)
    else:
        raise SystemExit(f"❌ HALT: Disabled API(s): {", ".join(disabled_svcs)}. Set ENABLE_APIS=True in Step 0 and re-run.")
print(f"✅ Required APIs ENABLED: {", ".join(REQUIRED_SERVICES)}")

# 2. Validate Apigee Organization, Environment, and Environment Group Hostname
APIGEE_BASE = f"https://apigee.googleapis.com/v1/organizations/{APIGEE_ORG}"
status, org_data = gcp_request("GET", APIGEE_BASE)
if status != 200 or org_data.get("state") != "ACTIVE":
    raise SystemExit(f"❌ HALT: Apigee org '{APIGEE_ORG}' not ACTIVE (HTTP {status}): {org_data}")

status, env_data = gcp_request("GET", f"{APIGEE_BASE}/environments/{APIGEE_ENV}")
if status != 200:
    raise SystemExit(f"❌ HALT: Apigee env '{APIGEE_ENV}' not found in org '{APIGEE_ORG}' (HTTP {status}): {env_data}")

_, eg_list = gcp_request("GET", f"{APIGEE_BASE}/envgroups")
matched_group, matched_hosts = None, []
for eg in eg_list.get("environmentGroups", []):
    eg_name = eg["name"]
    if ENV_GROUP and eg_name != ENV_GROUP:
        continue
    _, att_data = gcp_request("GET", f"{APIGEE_BASE}/envgroups/{eg_name}/attachments")
    if APIGEE_ENV in [a.get("environment") for a in att_data.get("environmentGroupAttachments", [])]:
        matched_group, matched_hosts = eg_name, eg.get("hostnames", [])
        break

if not matched_group or (not HOSTNAME and not matched_hosts):
    raise SystemExit(f"❌ HALT: Env '{APIGEE_ENV}' is not attached to a valid envgroup with a hostname in '{APIGEE_ORG}'.")
ENV_GROUP, HOSTNAME = matched_group, (HOSTNAME or matched_hosts[0])
os.environ.update({"ENV_GROUP": ENV_GROUP, "HOSTNAME": HOSTNAME})
print(f"✅ Apigee Topology: org={APIGEE_ORG} | env={APIGEE_ENV} | envgroup={ENV_GROUP} | host={HOSTNAME}")

# 3. Verify Effective Apigee IAM Permissions
REQUIRED_APIGEE_PERMS = [
    "apigee.proxies.create", "apigee.proxies.get", "apigee.proxies.update", "apigee.proxies.delete",
    "apigee.proxyrevisions.get", "apigee.proxyrevisions.update", "apigee.proxyrevisions.delete",
    "apigee.proxyrevisions.deploy", "apigee.proxyrevisions.undeploy",
    "apigee.deployments.create", "apigee.deployments.get", "apigee.deployments.update", "apigee.deployments.delete",
    "apigee.sharedflows.create", "apigee.sharedflows.get", "apigee.sharedflows.delete",
    "apigee.sharedflowrevisions.update", "apigee.sharedflowrevisions.deploy", "apigee.sharedflowrevisions.undeploy",
    "apigee.keyvaluemaps.create", "apigee.keyvaluemaps.get", "apigee.keyvaluemaps.update", "apigee.keyvaluemaps.delete",
    "apigee.keyvaluemapentries.create", "apigee.keyvaluemapentries.get", "apigee.keyvaluemapentries.update", "apigee.keyvaluemapentries.delete",
    "apigee.targetservers.create", "apigee.targetservers.get", "apigee.targetservers.update", "apigee.targetservers.delete",
]
iam_url = f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}:testIamPermissions"
_, iam_resp = gcp_request("POST", iam_url, {"permissions": REQUIRED_APIGEE_PERMS})
missing_perms = [p for p in REQUIRED_APIGEE_PERMS if p not in set(iam_resp.get("permissions", []))]
if missing_perms:
    raise SystemExit(f"❌ HALT: Missing Apigee IAM permissions: {", ".join(missing_perms)}")
PREFLIGHT["apigee"] = True
print(f"✅ [0.2] Verified all {len(REQUIRED_APIGEE_PERMS)} Apigee IAM permissions.")

# ============================================================================
# 0.3 Verify Service Account Roles, Permissions & <GoogleAccessToken> Bindings + Live Gemini
# ============================================================================
print("\n▶ [0.3] Verify Service Account Roles, Permissions & Bindings + Live Gemini...")
REQUIRED_SA_ROLES = ["roles/aiplatform.user", "roles/iam.serviceAccountTokenCreator"]

def ensure_policy_bindings(get_url: str, set_url: str, required_pairs: list, label: str):
    status, policy = gcp_request("POST", get_url, {})
    if status != 200:
        raise SystemExit(f"❌ HALT: Failed to read {label} IAM policy (HTTP {status}): {policy}")
    bindings = policy.setdefault("bindings", [])
    changed = False
    for member, role in required_pairs:
        entry = next((b for b in bindings if b.get("role") == role and not b.get("condition")), None)
        if entry and member in entry.get("members", []):
            print(f"  ✔ {label} verified: {role} -> {member}")
        else:
            print(f"  ⏳ Binding {label}: {role} -> {member}")
            if entry:
                entry.setdefault("members", []).append(member)
            else:
                bindings.append({"role": role, "members": [member]})
            changed = True
    if changed:
        s_code, s_resp = gcp_request("POST", set_url, {"policy": policy})
        if s_code != 200:
            raise SystemExit(f"❌ HALT: Failed to update {label} IAM policy (HTTP {s_code}): {s_resp}")
        print(f"  ✔ {label} bindings updated.")

# 1. Resolve Project Number & Apigee Service Agent
_, proj_data = gcp_request("GET", f"https://cloudresourcemanager.googleapis.com/v1/projects/{APIGEE_ORG}")
if not proj_data.get("projectNumber"):
    _, proj_data = gcp_request("GET", f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}")
if not proj_data.get("projectNumber"):
    raise SystemExit(f"❌ HALT: Unable to resolve projectNumber for '{PROJECT_ID}': {proj_data}")

PROJECT_NUMBER = str(proj_data["projectNumber"])
APIGEE_SERVICE_AGENT = f"service-{PROJECT_NUMBER}@gcp-sa-apigee.iam.gserviceaccount.com"

# 2. Verify Provided Service Account (or Auto-Provision Default Only When Not Provided)
target_sa = SERVICE_ACCOUNT or f"apigee-ai-gateway-sa@{PROJECT_ID}.iam.gserviceaccount.com"
sa_status, sa_data = gcp_request("GET", f"https://iam.googleapis.com/v1/projects/-/serviceAccounts/{target_sa}")

if SA_PROVIDED:
    if sa_status != 200 or sa_data.get("disabled", False):
        raise SystemExit(f"❌ HALT: Provided SERVICE_ACCOUNT '{SERVICE_ACCOUNT}' is missing or disabled (HTTP {sa_status}).")
    SA_STATUS = "VERIFIED_EXISTING"
    print(f"✅ Verified provided SERVICE_ACCOUNT: {SERVICE_ACCOUNT}")
elif sa_status == 200:
    SERVICE_ACCOUNT, SA_STATUS = target_sa, "REUSED_DEFAULT"
elif sa_status == 404:
    c_status, c_data = gcp_request(
        "POST",
        f"https://iam.googleapis.com/v1/projects/{PROJECT_ID}/serviceAccounts",
        {"accountId": "apigee-ai-gateway-sa", "serviceAccount": {"displayName": "Apigee AI Gateway Service Account"}},
    )
    if c_status not in (200, 201):
        raise SystemExit(f"❌ HALT: Failed to create default service account '{target_sa}' (HTTP {c_status}): {c_data}")
    SERVICE_ACCOUNT, SA_STATUS = c_data.get("email", target_sa), "CREATED_DEFAULT"
    time.sleep(3)
else:
    raise SystemExit(f"❌ HALT: Unable to inspect service account '{target_sa}' (HTTP {sa_status}): {sa_data}")

# 3. Verify & Ensure Project Roles (Gemini + Token Creator) and SA Resource Bindings (<GoogleAccessToken>)
proj_iam_base = f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}"
ensure_policy_bindings(
    f"{proj_iam_base}:getIamPolicy",
    f"{proj_iam_base}:setIamPolicy",
    [(f"serviceAccount:{SERVICE_ACCOUNT}", r) for r in REQUIRED_SA_ROLES],
    "Project role",
)

sa_iam_base = f"https://iam.googleapis.com/v1/projects/-/serviceAccounts/{SERVICE_ACCOUNT}"
ensure_policy_bindings(
    f"{sa_iam_base}:getIamPolicy",
    f"{sa_iam_base}:setIamPolicy",
    [
        (f"serviceAccount:{APIGEE_SERVICE_AGENT}", "roles/iam.serviceAccountTokenCreator"),
        (f"serviceAccount:{SERVICE_ACCOUNT}", "roles/iam.serviceAccountTokenCreator"),
        (f"serviceAccount:{SERVICE_ACCOUNT}", "roles/iam.serviceAccountUser"),
        (f"{caller_member_prefix}:{ACTIVE_IDENTITY}", "roles/iam.serviceAccountUser"),
    ],
    "SA <GoogleAccessToken> binding",
)

os.environ["SERVICE_ACCOUNT"] = SERVICE_ACCOUNT
PREFLIGHT["service_account"] = True
print(f"✅ Service Account ready for Gemini & <GoogleAccessToken>: {SERVICE_ACCOUNT} ({SA_STATUS})")

# 4. Verify Live Gemini Connectivity
payload = {"contents": [{"role": "user", "parts": [{"text": "Reply with READY."}]}], "generationConfig": {"maxOutputTokens": 8, "temperature": 0.0}}
def invoke_gemini(loc: str):
    host = "aiplatform.googleapis.com" if loc == "global" else f"{loc}-aiplatform.googleapis.com"
    url = f"https://{host}/v1/projects/{PROJECT_ID}/locations/{loc}/publishers/google/models/{GEMINI_MODEL}:generateContent"
    s, d = gcp_request("POST", url, payload, headers={"x-goog-user-project": PROJECT_ID})
    return gcp_request("POST", url, payload) if s == 403 else (s, d)

VERTEX_LOCATION, gen_status, gen_data = REGION, 0, {}
for _ in range(12):
    gen_status, gen_data = invoke_gemini(REGION)
    if gen_status == 200:
        VERTEX_LOCATION = REGION
        break
    for fb in [loc for loc in ("global", "us-central1") if loc != REGION]:
        fb_s, fb_d = invoke_gemini(fb)
        if fb_s == 200:
            gen_status, gen_data, VERTEX_LOCATION = fb_s, fb_d, fb
            break
    if gen_status == 200 or gen_status not in (403, 404):
        break
    time.sleep(5)

if gen_status != 200:
    raise SystemExit(f"❌ HALT: Live Gemini call failed (HTTP {gen_status}): {gen_data}")

os.environ["VERTEX_LOCATION"] = VERTEX_LOCATION
PREFLIGHT["gemini"] = True
print(f"✅ [0.3] Live Gemini verified: model={GEMINI_MODEL} | location={VERTEX_LOCATION}")

# ============================================================================
# 0.4 Install AFT CLI & Preflight Summary Gate
# ============================================================================
print("\n▶ [0.4] Install AFT CLI & Preflight Summary Gate...")
os.environ.pop("AFT_REPOSITORY", None)
for extra_bin in [str(pathlib.Path.home() / ".aft" / "bin"), str(pathlib.Path.home() / ".local" / "bin")]:
    if extra_bin not in os.environ["PATH"]:
        os.environ["PATH"] = f"{extra_bin}:{os.environ['PATH']}"

if not shutil.which("aft"):
    subprocess.run("curl -fsSL https://raw.githubusercontent.com/apigee/apigee-templater/main/install.sh | sh", shell=True, check=True)

aft_bin = shutil.which("aft")
if not aft_bin:
    raise SystemExit("❌ HALT: 'aft' binary not found on PATH after installation.")
aft_ver = subprocess.run([aft_bin, "--version"], capture_output=True, text=True, check=True).stdout.strip()
PREFLIGHT["aft"] = True
print(f"✅ AFT CLI ready: {aft_ver} ({aft_bin})")

# 2. Clone GitHub Repository (local templates & features for AFT)
REPO_URL = "https://github.com/luiscuellarh/apigee-unified-ai-gateway.git"
REPO_BRANCH = "main"
REPO_DIR = pathlib.Path.cwd() / "apigee-unified-ai-gateway"

if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", REPO_BRANCH], check=True, capture_output=True, text=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"], check=True, capture_output=True, text=True)
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True, capture_output=True, text=True)

if not (REPO_DIR / "templates").is_dir() or not (REPO_DIR / "features").is_dir():
    raise SystemExit(f"❌ HALT: Cloned repository at '{REPO_DIR}' is missing 'templates/' or 'features/' directories.")

WORKSPACE_DIR = REPO_DIR
os.environ.update({"REPO_URL": REPO_URL, "REPO_BRANCH": REPO_BRANCH, "REPO_DIR": str(REPO_DIR), "WORKSPACE_DIR": str(WORKSPACE_DIR)})
PREFLIGHT["repo"] = True
num_templates = len(list((REPO_DIR / "templates").glob("*.yaml")))
num_features = len(list((REPO_DIR / "features").glob("*.yaml")))
print(f"✅ Cloned GitHub repo ({REPO_BRANCH}): {REPO_DIR} ({num_templates} templates, {num_features} features)")

# 3. Preflight Summary Gate
failed_gates = [k for k, v in PREFLIGHT.items() if not v]
if failed_gates:
    raise SystemExit(f"❌ HALT: Preflight incomplete. Failed gate(s): {", ".join(failed_gates)}")

print("=" * 72)
for label, val in [
    ("GCP Project / No.",  f"{PROJECT_ID} ({PROJECT_NUMBER})"),
    ("Apigee Org / Env",   f"{APIGEE_ORG} / {APIGEE_ENV} ({ENV_GROUP})"),
    ("Gateway Hostname",   HOSTNAME),
    ("Active Identity",    ACTIVE_IDENTITY),
    ("Service Account",    f"{SERVICE_ACCOUNT} ({SA_STATUS}, <GoogleAccessToken> ready)"),
    ("Gemini Readiness",   f"PASS ({GEMINI_MODEL} @ {VERTEX_LOCATION})"),
    ("AFT Toolkit",        f"READY ({aft_ver})"),
    ("Local Repo (AFT)",   f"{REPO_DIR} (branch={REPO_BRANCH}, {num_templates} templates, {num_features} features)"),
]:
    print(f"{label:<20}: {val}")
print("=" * 72)
print("PREFLIGHT PASSED — safe to proceed")


## Step 1 — Deploy MCP Proxy for Loans, API Product & Developer App
- **Purpose:** Deploy the Model Context Protocol (MCP) Gateway proxy for the Loans service (`mcp-gateway-loan-product`), register the **AI Loan Agent Package** (`ai-loan-agent-package`) enforcing MCP operation-level allow-listing and quota (`tools/list` and `tools/call/calculate_monthly_installment` at 30 req/min), and onboard the **Loan Agent App** (`loan-agent@example.com`).
- **AFT Deployments:**
  1. **MCP Loan Proxy:** `aft -i templates/mcp-gateway-loan-product.yaml --organization $APIGEE_ORG --environment $APIGEE_ENVIRONMENT --sa $SERVICE_ACCOUNT`
  2. **API Product / Package:** `aft -i config/products/ai-loan-agent-package.yaml --organization $APIGEE_ORG --environment $APIGEE_ENVIRONMENT --sa $SERVICE_ACCOUNT`
  3. **Developer & App:** `aft -i config/apps/user_apps_mcp_loan.yaml --organization $APIGEE_ORG --environment $APIGEE_ENVIRONMENT --sa $SERVICE_ACCOUNT`
- **Dependencies:** Step 0 (Preflight)

In [ ]:
#@title 1.1 Deploy MCP Proxy for Loans, API Product & Developer App via AFT
# 1) Deploys templates/mcp-gateway-loan-product.yaml via AFT and waits until READY
# 2) Deploys config/products/ai-loan-agent-package.yaml via AFT
# 3) Deploys config/apps/user_apps_mcp_loan.yaml via AFT
# (Also ensures features/auth-oauth-server.yaml is active in APIGEE_ENVIRONMENT for OAuth 2.0 token generation)

os.environ.pop("AFT_REPOSITORY", None)
APIGEE_ENVIRONMENT = APIGEE_ENV
os.environ["APIGEE_ENVIRONMENT"] = APIGEE_ENVIRONMENT

if "wait_for_proxy_ready" not in globals():
    def wait_for_proxy_ready(proxy_name: str, env_name: str = None, timeout_sec: int = 180) -> str:
        env_target = env_name or APIGEE_ENV
        list_url = f"{APIGEE_BASE}/environments/{env_target}/apis/{proxy_name}/deployments"
        deadline = time.time() + timeout_sec
        last_rev, last_state = None, "NOT_DEPLOYED"
        print(f"⏳ Waiting for proxy '{proxy_name}' to reach READY state in env '{env_target}'...")
        while time.time() < deadline:
            s_list, d_list = gcp_request("GET", list_url)
            deployments = d_list.get("deployments", []) if s_list == 200 else []
            if deployments:
                latest = sorted(deployments, key=lambda d: int(d.get("revision", 0)))[-1]
                last_rev = latest.get("revision")
                rev_url = f"{APIGEE_BASE}/environments/{env_target}/apis/{proxy_name}/revisions/{last_rev}/deployments"
                s_rev, d_rev = gcp_request("GET", rev_url)
                last_state = d_rev.get("state", latest.get("state", "UNKNOWN")) if s_rev == 200 else "UNKNOWN"
                if last_state == "READY":
                    print(f"✅ Proxy '{proxy_name}' (revision {last_rev}) is READY in env '{env_target}'.")
                    return str(last_rev)
                print(f"   • proxy={proxy_name} | rev={last_rev} | state={last_state} — waiting 5s...")
            else:
                print(f"   • proxy={proxy_name} | state=WAITING_FOR_DEPLOYMENT — waiting 5s...")
            time.sleep(5)
        raise SystemExit(
            f"❌ HALT: Timed out waiting for proxy '{proxy_name}' to become READY in env '{env_target}' "
            f"(last revision={last_rev}, state={last_state})."
        )

TEMPLATE_LOAN_REL_PATH = "templates/mcp-gateway-loan-product.yaml"
FEATURE_LOAN_ENDPOINT_REL_PATH = "features/mcp-loan-endpoint.yaml"
FEATURE_OAUTH_QUOTA_REL_PATH = "features/api-quota-policy-spike-oauth.yaml"
FEATURE_OAUTH_SERVER_REL_PATH = "features/auth-oauth-server.yaml"
PRODUCT_LOAN_REL_PATH = "config/products/ai-loan-agent-package.yaml"
APPS_LOAN_REL_PATH = "config/apps/user_apps_mcp_loan.yaml"

required_rel_paths = [
    TEMPLATE_LOAN_REL_PATH,
    FEATURE_LOAN_ENDPOINT_REL_PATH,
    FEATURE_OAUTH_QUOTA_REL_PATH,
    FEATURE_OAUTH_SERVER_REL_PATH,
    PRODUCT_LOAN_REL_PATH,
    APPS_LOAN_REL_PATH,
]
if any(not (REPO_DIR / p).is_file() for p in required_rel_paths):
    print(f"🔄 Fetching latest files from origin/{REPO_BRANCH}...")
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH], capture_output=True, text=True)
    for rel_p in required_rel_paths:
        if not (REPO_DIR / rel_p).is_file():
            subprocess.run(["git", "-C", str(REPO_DIR), "checkout", f"origin/{REPO_BRANCH}", "--", rel_p], capture_output=True, text=True)

for rel_p in required_rel_paths:
    if not (REPO_DIR / rel_p).is_file():
        raise SystemExit(f"❌ HALT: Required file not found at '{REPO_DIR / rel_p}'. Ensure it is committed to the repo or re-run Step 0.")

# Ensure OAuth 2.0 token server (auth-oauth-server) is deployed in APIGEE_ENVIRONMENT so /v1/oauth/token can mint tokens
OAUTH_PROXY_NAME = "auth-oauth-server"
oauth_dep_url = f"{APIGEE_BASE}/environments/{APIGEE_ENVIRONMENT}/apis/{OAUTH_PROXY_NAME}/deployments"
oauth_s, oauth_d = gcp_request("GET", oauth_dep_url)
if oauth_s != 200 or not oauth_d.get("deployments"):
    print(f"⏳ Deploying OAuth 2.0 token server ({FEATURE_OAUTH_SERVER_REL_PATH}) to env '{APIGEE_ENVIRONMENT}'...")
    aft_oauth_cmd = [
        aft_bin,
        "-i", FEATURE_OAUTH_SERVER_REL_PATH,
        "--organization", APIGEE_ORG,
        "--environment", APIGEE_ENVIRONMENT,
        "--sa", SERVICE_ACCOUNT,
    ]
    print(f"⏳ Running: {' '.join(aft_oauth_cmd)}")
    oauth_res = subprocess.run(aft_oauth_cmd, cwd=str(REPO_DIR), capture_output=True, text=True)
    if oauth_res.returncode != 0:
        err_out = (oauth_res.stderr or oauth_res.stdout).strip()
        raise SystemExit(f"❌ HALT: AFT OAuth server deployment failed (exit {oauth_res.returncode}):\n{err_out}")
    if oauth_res.stdout.strip():
        print(oauth_res.stdout.strip())
    wait_for_proxy_ready(OAUTH_PROXY_NAME, APIGEE_ENVIRONMENT)
else:
    wait_for_proxy_ready(OAUTH_PROXY_NAME, APIGEE_ENVIRONMENT)

# 1. Deploy the MCP Loan Proxy
PROXY_NAME_MCP_LOAN = "mcp-gateway-loan-product"
aft_proxy_cmd = [
    aft_bin,
    "-i", TEMPLATE_LOAN_REL_PATH,
    "--organization", APIGEE_ORG,
    "--environment", APIGEE_ENVIRONMENT,
    "--sa", SERVICE_ACCOUNT,
]
print(f"\n▶ [1/3] Deploying MCP Loan Proxy: {' '.join(aft_proxy_cmd)}")
deploy_proxy_res = subprocess.run(aft_proxy_cmd, cwd=str(REPO_DIR), capture_output=True, text=True)
if deploy_proxy_res.returncode != 0:
    err_out = (deploy_proxy_res.stderr or deploy_proxy_res.stdout).strip()
    raise SystemExit(f"❌ HALT: AFT proxy deployment failed (exit {deploy_proxy_res.returncode}):\n{err_out}")
if deploy_proxy_res.stdout.strip():
    print(deploy_proxy_res.stdout.strip())

deployed_loan_rev = wait_for_proxy_ready(PROXY_NAME_MCP_LOAN, APIGEE_ENVIRONMENT)

# 2. Deploy the API Product / Package (ai-loan-agent-package)
prod_loan_path = REPO_DIR / PRODUCT_LOAN_REL_PATH
prod_loan_text = prod_loan_path.read_text(encoding="utf-8")
if not re.search(rf"(?m)^\s*-\s*{re.escape(APIGEE_ENVIRONMENT)}\s*$", prod_loan_text):
    prod_loan_text = re.sub(
        r"(?m)^(\s*environments:\s*\n)",
        rf"\g<1>  - {APIGEE_ENVIRONMENT}\n",
        prod_loan_text,
        count=1,
    )
    prod_loan_path.write_text(prod_loan_text, encoding="utf-8")

aft_prod_cmd = [
    aft_bin,
    "-i", PRODUCT_LOAN_REL_PATH,
    "--organization", APIGEE_ORG,
    "--environment", APIGEE_ENVIRONMENT,
    "--sa", SERVICE_ACCOUNT,
]
print(f"\n▶ [2/3] Deploying API Product / Package: {' '.join(aft_prod_cmd)}")
deploy_prod_res = subprocess.run(aft_prod_cmd, cwd=str(REPO_DIR), capture_output=True, text=True)
if deploy_prod_res.returncode != 0:
    err_out = (deploy_prod_res.stderr or deploy_prod_res.stdout).strip()
    raise SystemExit(f"❌ HALT: AFT product deployment failed (exit {deploy_prod_res.returncode}):\n{err_out}")
if deploy_prod_res.stdout.strip():
    print(deploy_prod_res.stdout.strip())
print(f"✅ Deployed API Product: {PRODUCT_LOAN_REL_PATH}")

# 3. Deploy the Developer and App (user_apps_mcp_loan.yaml)
aft_apps_cmd = [
    aft_bin,
    "-i", APPS_LOAN_REL_PATH,
    "--organization", APIGEE_ORG,
    "--environment", APIGEE_ENVIRONMENT,
    "--sa", SERVICE_ACCOUNT,
]
print(f"\n▶ [3/3] Deploying Developer & App: {' '.join(aft_apps_cmd)}")
deploy_apps_res = subprocess.run(aft_apps_cmd, cwd=str(REPO_DIR), capture_output=True, text=True)
if deploy_apps_res.returncode != 0:
    err_out = (deploy_apps_res.stderr or deploy_apps_res.stdout).strip()
    raise SystemExit(f"❌ HALT: AFT developer/app deployment failed (exit {deploy_apps_res.returncode}):\n{err_out}")
if deploy_apps_res.stdout.strip():
    print(deploy_apps_res.stdout.strip())
print(f"✅ Deployed Developer & App: {APPS_LOAN_REL_PATH}")

OAUTH_TOKEN_ENDPOINT = f"https://{HOSTNAME}/v1/oauth/token"
MCP_LOAN_ENDPOINT = f"https://{HOSTNAME}/loanservice/mcp"
os.environ.update({
    "PROXY_NAME_MCP_LOAN": PROXY_NAME_MCP_LOAN,
    "OAUTH_TOKEN_ENDPOINT": OAUTH_TOKEN_ENDPOINT,
    "MCP_LOAN_ENDPOINT": MCP_LOAN_ENDPOINT,
})

print("=" * 72)
print(f"✅ Deployed MCP Proxy : {PROXY_NAME_MCP_LOAN} (rev={deployed_loan_rev})")
print(f"✅ API Product        : ai-loan-agent-package (tools/list, tools/call/calculate_monthly_installment)")
print(f"✅ Developer App      : Loan Agent App (loan-agent@example.com)")
print(f"✅ OAuth Token URL    : {OAUTH_TOKEN_ENDPOINT}")
print(f"✅ MCP Loan Endpoint  : {MCP_LOAN_ENDPOINT}")
print("=" * 72)

In [ ]:
#@title 1.2 Test MCP Loan Proxy (Standard MCP Protocol & Dynamic Tool UX)
#@markdown ### OAuth 2.0 App Credentials (`loan-agent-app`)
CLIENT_ID = "loan-agent-app-key-123" #@param {type:"string"}
CLIENT_SECRET = "loan-agent-app-secret-123" #@param {type:"string"}

import json
import subprocess
import time
from IPython.display import display, HTML
import ipywidgets as widgets
import requests
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# ==============================================================================
# 0. Validate Prerequisite State & Ensure Proxy Revision is READY
# ==============================================================================
if "HOSTNAME" not in globals() or not str(HOSTNAME).strip():
  raise RuntimeError("❌ HOSTNAME is empty. Please run Step 0 first.")

_org = str(globals().get("APIGEE_ORG", "")).strip()
_env = str(globals().get("APIGEE_ENV", "")).strip()
_proxy_name = "mcp-gateway-loan-product"

def _wait_for_proxy_ready(org: str, env: str, proxy_name: str, timeout_s: int = 120):
  if not org or not env:
    return
  tok_res = subprocess.run(
      ["gcloud", "auth", "print-access-token"],
      capture_output=True, text=True, check=False
  )
  if tok_res.returncode != 0 or not tok_res.stdout.strip():
    return
  hdrs = {"Authorization": f"Bearer {tok_res.stdout.strip()}"}
  dep_url = f"https://apigee.googleapis.com/v1/organizations/{org}/environments/{env}/apis/{proxy_name}/deployments"
  r_dep = requests.get(dep_url, headers=hdrs, timeout=15)
  if r_dep.status_code != 200:
    return
  deployments = r_dep.json().get("deployments", [])
  if not deployments:
    return
  rev = deployments[0].get("revision")
  if not rev:
    return
  rev_url = f"https://apigee.googleapis.com/v1/organizations/{org}/environments/{env}/apis/{proxy_name}/revisions/{rev}/deployments"
  print(f"⏳ Waiting for proxy '{proxy_name}' (revision {rev}) to reach READY state in env '{env}'...")
  deadline = time.time() + timeout_s
  while time.time() < deadline:
    r_state = requests.get(rev_url, headers=hdrs, timeout=15)
    if r_state.status_code == 200:
      state = r_state.json().get("state", "")
      if state == "READY":
        print(f"✅ Proxy '{proxy_name}' (revision {rev}) is READY in env '{env}'.")
        return
    time.sleep(3)
  print(f"⚠️ Timed out waiting for proxy '{proxy_name}' READY state; proceeding anyway.")

_wait_for_proxy_ready(_org, _env, _proxy_name)

_host = str(HOSTNAME).strip()
OAUTH_TOKEN_URL = f"https://{_host}/v1/oauth/token"
MCP_ENDPOINT_URL = f"https://{_host}/loanservice/mcp"

print(f"🔐 OAuth Token Endpoint : {OAUTH_TOKEN_URL}")
print(f"🌐 MCP Loan Endpoint    : {MCP_ENDPOINT_URL}")
print(f"🆔 Client ID            : {CLIENT_ID}")

# ==============================================================================
# 1. Standard MCP Streamable HTTP Client
#    Implements the Model Context Protocol (MCP) specification:
#      1) OAuth 2.0 client_credentials token minting
#      2) initialize -> negotiates protocolVersion & captures mcp-session-id
#      3) notifications/initialized -> completes session handshake
#      4) tools/list -> dynamically discovers available tools & inputSchema
#      5) tools/call -> invokes selected tool with schema-typed arguments dict
# ==============================================================================
class StandardMCPClient:
  """MCP Streamable HTTP client implementing the standard MCP lifecycle."""

  def __init__(self, oauth_url: str, mcp_url: str, client_id: str, client_secret: str):
    self.oauth_url = oauth_url
    self.mcp_url = mcp_url
    self.client_id = client_id
    self.client_secret = client_secret
    self.access_token = None
    self.session_id = None
    self.protocol_version = "2024-11-05"
    self.server_info = {}
    self.server_capabilities = {}
    self._rpc_id = 0
    self.http = requests.Session()
    self.http.verify = False

  def _next_id(self) -> int:
    self._rpc_id += 1
    return self._rpc_id

  def mint_oauth_token(self) -> str:
    t0 = time.time()
    resp = self.http.post(
        self.oauth_url,
        auth=(self.client_id, self.client_secret),
        headers={"Content-Type": "application/x-www-form-urlencoded"},
        data={"grant_type": "client_credentials"},
        timeout=20,
    )
    ms = int((time.time() - t0) * 1000)
    if resp.status_code != 200:
      raise RuntimeError(
          f"❌ Failed to mint OAuth token (HTTP {resp.status_code}, {ms}ms): {resp.text[:500]}"
      )
    data = resp.json()
    self.access_token = data.get("access_token", "")
    expires_in = data.get("expires_in", "N/A")
    masked = (
        f"{self.access_token[:8]}...{self.access_token[-6:]}"
        if len(self.access_token) > 14 else "***"
    )
    print(f"✅ [1/4] OAuth 2.0 Token Minted (HTTP 200 | {ms}ms | token={masked} | expires_in={expires_in}s)")
    return self.access_token

  def _headers(self, include_session: bool = True) -> dict:
    hdrs = {
        "Authorization": f"Bearer {self.access_token}",
        "Content-Type": "application/json",
        "Accept": "application/json, text/event-stream",
    }
    if include_session and self.session_id:
      hdrs["mcp-session-id"] = self.session_id
      hdrs["mcp-protocol-version"] = self.protocol_version
    return hdrs

  @staticmethod
  def _parse_mcp_response(resp: requests.Response) -> dict:
    """Parses either application/json or text/event-stream (SSE) MCP responses."""
    ctype = (resp.headers.get("Content-Type") or "").lower()
    raw_text = (resp.text or "").strip()
    if not raw_text:
      return {}
    if "text/event-stream" in ctype or raw_text.startswith("event:") or raw_text.startswith("data:"):
      last_json = {}
      for line in raw_text.splitlines():
        line_s = line.strip()
        if line_s.startswith("data:"):
          data_str = line_s[len("data:"):].strip()
          if data_str:
            try:
              last_json = json.loads(data_str)
            except Exception:
              pass
      if last_json:
        return last_json
    try:
      return resp.json()
    except Exception:
      return {"raw_text": raw_text}

  def _capture_session_id(self, resp: requests.Response):
    for k, v in resp.headers.items():
      if k.lower() == "mcp-session-id" and v:
        self.session_id = v.strip()
        break

  def initialize_session(self) -> dict:
    if not self.access_token:
      self.mint_oauth_token()

    # Step A: Standard MCP initialize request
    self.session_id = None
    init_payload = {
        "jsonrpc": "2.0",
        "id": self._next_id(),
        "method": "initialize",
        "params": {
            "protocolVersion": self.protocol_version,
            "capabilities": {},
            "clientInfo": {
                "name": "apigee-mcp-colab-client",
                "version": "1.0.0",
            },
        },
    }
    t0 = time.time()
    resp = self.http.post(
        self.mcp_url,
        headers=self._headers(include_session=False),
        json=init_payload,
        timeout=30,
    )
    ms = int((time.time() - t0) * 1000)
    self._capture_session_id(resp)
    body = self._parse_mcp_response(resp)

    if resp.status_code != 200 or "error" in body:
      raise RuntimeError(
          f"❌ MCP initialize failed (HTTP {resp.status_code}, {ms}ms): {json.dumps(body, ensure_ascii=False)}"
      )

    result = body.get("result", {})
    self.protocol_version = result.get("protocolVersion", self.protocol_version)
    self.server_info = result.get("serverInfo", {})
    self.server_capabilities = result.get("capabilities", {})

    srv_name = self.server_info.get("name", "unknown")
    srv_ver = self.server_info.get("version", "unknown")
    print(
        f"✅ [2/4] MCP 'initialize' Succeeded (HTTP {resp.status_code} | {ms}ms | "
        f"server={srv_name} v{srv_ver} | protocol={self.protocol_version} | "
        f"mcp-session-id={self.session_id or 'stateless'})"
    )

    # Step B: Standard MCP notifications/initialized notification
    notif_payload = {
        "jsonrpc": "2.0",
        "method": "notifications/initialized",
    }
    t1 = time.time()
    notif_resp = self.http.post(
        self.mcp_url,
        headers=self._headers(include_session=True),
        json=notif_payload,
        timeout=20,
    )
    notif_ms = int((time.time() - t1) * 1000)
    self._capture_session_id(notif_resp)
    print(
        f"✅ [3/4] MCP 'notifications/initialized' Sent "
        f"(HTTP {notif_resp.status_code} | {notif_ms}ms)"
    )
    return result

  def list_tools(self) -> list:
    """Executes standard MCP tools/list over the initialized MCP session."""
    if not self.session_id and not self.server_info:
      self.initialize_session()

    payload = {
        "jsonrpc": "2.0",
        "id": self._next_id(),
        "method": "tools/list",
    }
    t0 = time.time()
    resp = self.http.post(
        self.mcp_url,
        headers=self._headers(include_session=True),
        json=payload,
        timeout=30,
    )
    ms = int((time.time() - t0) * 1000)
    self._capture_session_id(resp)
    body = self._parse_mcp_response(resp)

    if resp.status_code != 200 or "error" in body:
      raise RuntimeError(
          f"❌ MCP tools/list failed (HTTP {resp.status_code}, {ms}ms): {json.dumps(body, ensure_ascii=False)}"
      )

    tools = body.get("result", {}).get("tools", [])
    print(
        f"✅ [4/4] MCP 'tools/list' Succeeded "
        f"(HTTP {resp.status_code} | {ms}ms | discovered {len(tools)} tool(s))"
    )
    return tools

  def call_tool(self, name: str, arguments: dict) -> tuple:
    """Executes standard MCP tools/call with automatic session recovery if expired."""
    if not isinstance(arguments, dict):
      raise ValueError(
          f"MCP tools/call 'arguments' must be a dict (JSON object), got {type(arguments).__name__}"
      )

    payload = {
        "jsonrpc": "2.0",
        "id": self._next_id(),
        "method": "tools/call",
        "params": {
            "name": name,
            "arguments": arguments,
        },
    }
    t0 = time.time()
    resp = self.http.post(
        self.mcp_url,
        headers=self._headers(include_session=True),
        json=payload,
        timeout=30,
    )
    ms = int((time.time() - t0) * 1000)
    body = self._parse_mcp_response(resp)

    # If the Cloud Run MCP session expired between interactive clicks, re-initialize once
    err_msg = str(body.get("error", {}).get("message", "")).lower() if isinstance(body, dict) else ""
    if resp.status_code == 404 or "before initialization" in err_msg or "session" in err_msg:
      print("🔄 MCP session expired on server; re-running MCP handshake automatically...")
      self.initialize_session()
      t0 = time.time()
      resp = self.http.post(
          self.mcp_url,
          headers=self._headers(include_session=True),
          json=payload,
          timeout=30,
      )
      ms = int((time.time() - t0) * 1000)
      body = self._parse_mcp_response(resp)

    return payload, resp, body, ms


# ==============================================================================
# 2. Automatic MCP Handshake & Dynamic Tool Discovery (tools/list)
# ==============================================================================
mcp_client = StandardMCPClient(
    oauth_url=OAUTH_TOKEN_URL,
    mcp_url=MCP_ENDPOINT_URL,
    client_id=CLIENT_ID.strip(),
    client_secret=CLIENT_SECRET.strip(),
)

print("\n🚀 Running Standard MCP Protocol Handshake & Automatic `tools/list` Discovery...")
print("-" * 80)
mcp_client.mint_oauth_token()
mcp_client.initialize_session()
discovered_tools = mcp_client.list_tools()
print("-" * 80)

TOOLS_BY_NAME = {t.get("name"): t for t in discovered_tools if t.get("name")}

if discovered_tools:
  print("\n🧰 Discovered MCP Tools & Input Schemas from `tools/list`:")
  for idx, t in enumerate(discovered_tools, start=1):
    t_name = t.get("name", "unnamed")
    t_desc = (t.get("description") or "No description").strip()
    schema = t.get("inputSchema") or {}
    props = schema.get("properties") or {}
    req_fields = set(schema.get("required") or [])
    param_summaries = []
    for p_name, p_spec in props.items():
      p_type = p_spec.get("type", "any")
      req_marker = "*" if p_name in req_fields else ""
      param_summaries.append(f"{p_name}{req_marker}: {p_type}")
    params_str = ", ".join(param_summaries) if param_summaries else "no parameters"
    print(f"  {idx}. {t_name}({params_str})")
    print(f"     └─ {t_desc.splitlines()[0]}")
else:
  print("⚠️ No tools returned by `tools/list`.")

# ==============================================================================
# 3. Dynamic Interactive MCP Client UX (Built Strictly from `tools/list` Schema)
# ==============================================================================
tool_options = [
    (f"{t['name']}", t["name"]) for t in discovered_tools if t.get("name")
]

tool_dropdown = widgets.Dropdown(
    options=tool_options,
    value=tool_options[0][1] if tool_options else None,
    description="MCP Tool:",
    style={"description_width": "110px"},
    layout=widgets.Layout(width="480px"),
)

tool_info_html = widgets.HTML()
params_container = widgets.VBox(layout=widgets.Layout(margin="8px 0px 8px 0px"))
call_button = widgets.Button(
    description="▶ Execute tools/call",
    button_style="primary",
    layout=widgets.Layout(width="220px", height="36px"),
)
refresh_button = widgets.Button(
    description="🔄 Refresh tools/list",
    button_style="info",
    layout=widgets.Layout(width="200px", height="36px"),
)
output_area = widgets.Output(
    layout=widgets.Layout(
        border="1px solid #dadce0",
        padding="12px",
        margin="10px 0px 0px 0px",
    )
)

# Maps parameter name -> (widget, property_schema, is_required)
_current_param_widgets = {}


def _default_value_for_prop(prop_schema: dict):
  """Determines initial widget value strictly from the tool's inputSchema."""
  if "default" in prop_schema and prop_schema["default"] is not None:
    return prop_schema["default"]
  if "enum" in prop_schema and prop_schema["enum"]:
    return prop_schema["enum"][0]

  p_type = prop_schema.get("type", "string")
  if isinstance(p_type, list):
    p_type = next((x for x in p_type if x != "null"), "string")

  if p_type == "boolean":
    return False
  if p_type == "integer":
    return 0
  if p_type == "number":
    return 0.0
  if p_type == "array":
    return "[]"
  if p_type == "object":
    return "{}"
  return ""


def _build_param_widgets_for_tool(tool_name: str):
  """Dynamically generates input widgets from the selected tool's inputSchema."""
  _current_param_widgets.clear()
  if not tool_name or tool_name not in TOOLS_BY_NAME:
    tool_info_html.value = "<i>No tool selected.</i>"
    params_container.children = []
    return

  tool_def = TOOLS_BY_NAME[tool_name]
  desc = (tool_def.get("description") or "No description provided by MCP server.").strip()
  schema = tool_def.get("inputSchema") or {}
  props = schema.get("properties") or {}
  required_set = set(schema.get("required") or [])

  tool_info_html.value = (
      "<div style='background:#f1f8ff;border-left:4px solid #1a73e8;padding:8px 12px;margin:4px 0;'>"
      f"<b>Tool:</b> <code>{tool_name}</code><br>"
      f"<b>Description:</b> {desc}<br>"
      f"<b>Schema Parameters:</b> {len(props)} property(ies) "
      f"({len(required_set)} required)"
      "</div>"
  )

  if not props:
    params_container.children = [
        widgets.HTML("<i>This tool takes no input parameters (<code>arguments: {}</code>).</i>")
    ]
    return

  rows = []
  for p_name, p_spec in props.items():
    is_req = p_name in required_set
    p_type = p_spec.get("type", "string")
    if isinstance(p_type, list):
      p_type = next((x for x in p_type if x != "null"), "string")
    p_desc = p_spec.get("description") or p_spec.get("title") or ""
    default_val = _default_value_for_prop(p_spec)

    label_str = f"{p_name}{' *' if is_req else ''} ({p_type}):"
    w_style = {"description_width": "220px"}
    w_layout = widgets.Layout(width="460px")

    if "enum" in p_spec and p_spec["enum"]:
      w = widgets.Dropdown(
          options=p_spec["enum"],
          value=default_val,
          description=label_str,
          style=w_style,
          layout=w_layout,
      )
    elif p_type == "boolean":
      w = widgets.Checkbox(
          value=bool(default_val),
          description=label_str,
          style=w_style,
          layout=w_layout,
      )
    elif p_type == "integer":
      w = widgets.IntText(
          value=int(default_val) if default_val != "" else 0,
          description=label_str,
          style=w_style,
          layout=w_layout,
      )
    elif p_type == "number":
      w = widgets.FloatText(
          value=float(default_val) if default_val != "" else 0.0,
          description=label_str,
          style=w_style,
          layout=w_layout,
      )
    elif p_type in ("array", "object"):
      w = widgets.Textarea(
          value=default_val if isinstance(default_val, str) else json.dumps(default_val),
          description=label_str,
          placeholder=f"Enter valid JSON {p_type}",
          style=w_style,
          layout=widgets.Layout(width="460px", height="60px"),
      )
    else:
      w = widgets.Text(
          value=str(default_val),
          description=label_str,
          placeholder=p_desc or f"Enter {p_name} ({p_type})",
          style=w_style,
          layout=w_layout,
      )

    _current_param_widgets[p_name] = (w, p_spec, is_req)
    help_html = widgets.HTML(
        f"<span style='color:#5f6368;font-size:12px;margin-left:8px;'>"
        f"{'<b>[required]</b> ' if is_req else '[optional] '}{p_desc}</span>"
    )
    rows.append(widgets.HBox([w, help_html], layout=widgets.Layout(align_items="center")))

  params_container.children = rows


def _collect_arguments_from_widgets(tool_name: str) -> tuple:
  """Reads values from the dynamically generated widgets and returns (tool_name, arguments_dict)."""
  if not tool_name:
    raise ValueError("No MCP tool available/selected.")

  args = {}
  for p_name, (w, p_spec, is_req) in _current_param_widgets.items():
    p_type = p_spec.get("type", "string")
    if isinstance(p_type, list):
      p_type = next((x for x in p_type if x != "null"), "string")
    val = w.value

    if p_type in ("array", "object"):
      raw = str(val).strip()
      if not raw and not is_req:
        continue
      args[p_name] = json.loads(raw)
    elif p_type == "string":
      if val == "" and not is_req:
        continue
      args[p_name] = str(val)
    elif p_type == "integer":
      args[p_name] = int(val)
    elif p_type == "number":
      args[p_name] = float(val)
    elif p_type == "boolean":
      args[p_name] = bool(val)
    else:
      args[p_name] = val

  return tool_name, args


def _on_tool_selection_change(change):
  if change["name"] == "value":
    _build_param_widgets_for_tool(change["new"])


def _on_refresh_clicked(_):
  with output_area:
    output_area.clear_output()
    print("🔄 Re-running MCP Handshake & `tools/list`...")
    try:
      mcp_client.initialize_session()
      fresh_tools = mcp_client.list_tools()
      TOOLS_BY_NAME.clear()
      for t in fresh_tools:
        if t.get("name"):
          TOOLS_BY_NAME[t["name"]] = t
      new_opts = [(f"{t['name']}", t["name"]) for t in fresh_tools if t.get("name")]
      tool_dropdown.options = new_opts
      if new_opts:
        tool_dropdown.value = new_opts[0][1]
        _build_param_widgets_for_tool(tool_dropdown.value)
      print(f"✅ Refreshed {len(fresh_tools)} tool(s) from MCP server.")
    except Exception as exc:
      print(f"❌ Error refreshing tools/list: {exc}")


def _on_call_clicked(_):
  with output_area:
    output_area.clear_output()
    try:
      target_tool, tool_args = _collect_arguments_from_widgets(tool_dropdown.value)
    except Exception as exc:
      print(f"❌ Parameter Validation Error: {exc}")
      return

    try:
      req_payload, resp, body, ms = mcp_client.call_tool(target_tool, tool_args)
    except Exception as exc:
      print(f"❌ MCP tools/call failed: {exc}")
      return

    print("📤 MCP JSON-RPC 2.0 Request Payload:")
    print(json.dumps(req_payload, indent=2, ensure_ascii=False))
    print("-" * 72)

    if resp.status_code == 200:
      print(
          f"✅ ALLOWED BY MCP GATEWAY "
          f"(HTTP 200 | tool={target_tool} | session={mcp_client.session_id} | latency={ms}ms)"
      )
      if "error" in body:
        print("\n⚠️ MCP Server returned a JSON-RPC error:")
        print(json.dumps(body["error"], indent=2, ensure_ascii=False))
      else:
        result = body.get("result", body)
        is_err = result.get("isError", False) if isinstance(result, dict) else False
        content_items = result.get("content", []) if isinstance(result, dict) else []
        if content_items:
          header_icon = "⚠️" if is_err else "🎯"
          print(f"\n{header_icon} MCP Tool Execution Output (`result.content`):")
          for item in content_items:
            if isinstance(item, dict) and item.get("type") == "text":
              text_val = item.get("text", "")
              try:
                parsed_inner = json.loads(text_val)
                print(json.dumps(parsed_inner, indent=2, ensure_ascii=False))
              except Exception:
                print(text_val)
            else:
              print(json.dumps(item, indent=2, ensure_ascii=False))
        print("\n📦 Full JSON-RPC 2.0 Response:")
        print(json.dumps(body, indent=2, ensure_ascii=False))
    elif resp.status_code in (401, 403):
      print(
          f"🛡️ BLOCKED BY APIGEE MCP GATEWAY "
          f"(HTTP {resp.status_code} | tool={target_tool} | latency={ms}ms)"
      )
      print("\n📦 Gateway Policy Enforcement Response:")
      print(json.dumps(body, indent=2, ensure_ascii=False))
    else:
      print(f"⚠️ Unexpected HTTP {resp.status_code} ({ms}ms):")
      print(json.dumps(body, indent=2, ensure_ascii=False))


tool_dropdown.observe(_on_tool_selection_change, names="value")
call_button.on_click(_on_call_clicked)
refresh_button.on_click(_on_refresh_clicked)

# Build dynamic parameter inputs for the first discovered tool
_build_param_widgets_for_tool(tool_dropdown.value)

display(
    HTML(
        "<h4 style='margin:12px 0 6px 0;'>🎛️ Interactive MCP Client "
        "(Dynamically Generated from <code>tools/list</code>)</h4>"
    ),
    widgets.VBox([
        widgets.HBox([tool_dropdown, refresh_button]),
        tool_info_html,
        params_container,
        call_button,
        output_area,
    ]),
)
